In [35]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    def __repr__(self):
        return f"Value (data={self.data}, grad={self.grad}, op={self._op})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out 

In [37]:
a, b, c = Value(2.0), Value(3.0), Value(4.0)
ab = a * b
print(ab)
L = ab + c
L.grad = 1.0
L._backward()
ab._backward()
print("a.grad, b.grad, c.grad:", a.grad, b.grad, c.grad)

# Test 2: same variable used twice
x = Value(3.0)
y = x * x
y.grad = 1.0
y._backward()
print("x.grad:", x.grad)

Value (data=6.0, grad=0.0, op=*)
a.grad, b.grad, c.grad: 3.0 2.0 1.0
x.grad: 6.0
